# Machine Learning Adaptive SuperTrend 指标实现

本Notebook将Pine Script的Machine Learning Adaptive SuperTrend指标转换为Python实现，并使用QQQ数据进行测试。

## 目标
1. 分析Pine Script源代码
2. 转换为Python实现
3. 使用QQQ 2023-2025数据测试
4. 在K线图上可视化结果

---

## 1. 导入必要的库

In [ ]:
# 数据处理和计算
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 数据获取
import yfinance as yf

# 可视化
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# 机器学习
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 技术指标
import pandas_ta as ta

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8')

# 显示所有列
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("✅ 库导入完成")

## 2. Pine Script 代码分析区域

**请在下方单元格中粘贴您的Pine Script源代码：**

In [ ]:
# Pine Script 源代码将在这里分析
pine_script_code = """
请在这里粘贴您的Pine Script代码
"""

print("Pine Script代码已准备好进行分析")
print("代码长度:", len(pine_script_code))

## 3. 数据获取和预处理

In [ ]:
def get_stock_data(symbol='QQQ', start_date='2023-01-01', end_date='2025-01-01'):
    """
    获取股票数据
    """
    try:
        # 下载数据
        ticker = yf.Ticker(symbol)
        data = ticker.history(start=start_date, end=end_date, interval='1d')
        
        # 重命名列以匹配标准格式
        data = data.rename(columns={
            'Open': 'open',
            'High': 'high', 
            'Low': 'low',
            'Close': 'close',
            'Volume': 'volume'
        })
        
        # 删除不需要的列
        columns_to_keep = ['open', 'high', 'low', 'close', 'volume']
        data = data[columns_to_keep]
        
        # 去除缺失值
        data = data.dropna()
        
        print(f"✅ 成功获取 {symbol} 数据")
        print(f"数据期间: {data.index[0].date()} 到 {data.index[-1].date()}")
        print(f"数据条数: {len(data)}")
        print(f"\n数据预览:")
        print(data.head())
        
        return data
        
    except Exception as e:
        print(f"❌ 数据获取失败: {str(e)}")
        return None

# 获取QQQ数据
qqq_data = get_stock_data('QQQ', '2023-01-01', '2025-01-01')

## 4. ML Adaptive SuperTrend 指标实现框架

基于Pine Script代码分析，我们将实现以下组件：
1. 基础技术指标计算 (ATR, EMA, etc.)
2. 机器学习特征工程
3. 自适应参数优化
4. SuperTrend计算

In [ ]:
class MLAdaptiveSuperTrend:
    """
    Machine Learning Adaptive SuperTrend 指标类
    """
    
    def __init__(self, period=14, multiplier=3.0, ml_lookback=50):
        self.period = period
        self.multiplier = multiplier
        self.ml_lookback = ml_lookback
        self.model = None
        
    def calculate_atr(self, high, low, close, period=14):
        """
        计算平均真实波幅 (ATR)
        """
        try:
            # 使用pandas-ta计算ATR
            atr = ta.atr(high=high, low=low, close=close, length=period)
            return atr.fillna(method='ffill')
        except:
            # 备用计算方法
            high_low = high - low
            high_close_prev = np.abs(high - close.shift(1))
            low_close_prev = np.abs(low - close.shift(1))
            
            true_range = np.maximum(high_low, 
                                  np.maximum(high_close_prev, low_close_prev))
            atr = true_range.rolling(window=period).mean()
            return atr
    
    def calculate_hl2(self, high, low):
        """
        计算HL2 (典型价格的简化版本)
        """
        return (high + low) / 2
    
    def prepare_ml_features(self, data):
        """
        准备机器学习特征
        """
        features = pd.DataFrame(index=data.index)
        
        # 价格特征
        features['rsi'] = ta.rsi(data['close'], length=14)
        macd_data = ta.macd(data['close'])
        features['macd'] = macd_data['MACD_12_26_9'] if 'MACD_12_26_9' in macd_data.columns else macd_data.iloc[:,0]
        bb_data = ta.bbands(data['close'])
        features['bb_upper'] = bb_data['BBU_20_2.0'] if 'BBU_20_2.0' in bb_data.columns else bb_data.iloc[:,0]
        features['bb_middle'] = bb_data['BBM_20_2.0'] if 'BBM_20_2.0' in bb_data.columns else bb_data.iloc[:,1]
        features['bb_lower'] = bb_data['BBL_20_2.0'] if 'BBL_20_2.0' in bb_data.columns else bb_data.iloc[:,2]
        
        # 波动性特征
        features['atr'] = self.calculate_atr(data['high'], data['low'], data['close'])
        features['volatility'] = data['close'].rolling(20).std()
        
        # 趋势特征
        features['sma_5'] = data['close'].rolling(5).mean()
        features['sma_20'] = data['close'].rolling(20).mean()
        features['ema_12'] = data['close'].ewm(span=12).mean()
        features['ema_26'] = data['close'].ewm(span=26).mean()
        
        # 价格位置特征
        features['price_position'] = (data['close'] - data['low'].rolling(20).min()) / \
                                    (data['high'].rolling(20).max() - data['low'].rolling(20).min())
        
        # 成交量特征
        features['volume_sma'] = data['volume'].rolling(20).mean()
        features['volume_ratio'] = data['volume'] / features['volume_sma']
        
        return features.fillna(method='ffill').fillna(0)
    
    def fit_ml_model(self, features, target):
        """
        训练机器学习模型来预测最优参数
        """
        # 清理数据
        valid_idx = ~(np.isnan(target) | np.isinf(target))
        features_clean = features[valid_idx]
        target_clean = target[valid_idx]
        
        if len(features_clean) < 10:
            print("⚠️ 数据不足，使用默认参数")
            return None
            
        # 标准化特征
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features_clean)
        
        # 训练随机森林模型
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(features_scaled, target_clean)
        
        return {'model': model, 'scaler': scaler}
    
    def predict_adaptive_multiplier(self, features, ml_model):
        """
        使用机器学习模型预测自适应乘数
        """
        if ml_model is None:
            return pd.Series([self.multiplier] * len(features), index=features.index)
            
        try:
            features_scaled = ml_model['scaler'].transform(features)
            predictions = ml_model['model'].predict(features_scaled)
            
            # 限制预测范围在合理区间内
            predictions = np.clip(predictions, 1.0, 5.0)
            
            return pd.Series(predictions, index=features.index)
        except Exception as e:
            print(f"⚠️ 预测失败，使用默认乘数: {str(e)}")
            return pd.Series([self.multiplier] * len(features), index=features.index)
    
    def calculate_supertrend(self, data):
        """
        计算ML Adaptive SuperTrend
        """
        print("🔄 开始计算ML Adaptive SuperTrend...")
        
        # 准备基础数据
        hl2 = self.calculate_hl2(data['high'], data['low'])
        atr = self.calculate_atr(data['high'], data['low'], data['close'], self.period)
        
        # 准备机器学习特征
        features = self.prepare_ml_features(data)
        
        # 初始化输出Series
        supertrend_upper = pd.Series(np.nan, index=data.index)
        supertrend_lower = pd.Series(np.nan, index=data.index)
        supertrend = pd.Series(np.nan, index=data.index)
        trend = pd.Series(1, index=data.index)
        adaptive_multiplier = pd.Series(self.multiplier, index=data.index)
        
        # 等待足够的数据进行ML训练
        if len(data) >= self.ml_lookback + self.period:
            print(f"📊 训练机器学习模型 (lookback={self.ml_lookback})...")
            
            # 使用历史波动率作为训练目标
            historical_volatility = data['close'].rolling(self.period).std() / data['close']
            target_multiplier = 2.0 + historical_volatility * 10  # 基于波动率调整乘数
            
            # 训练模型
            train_start = self.ml_lookback
            ml_model = self.fit_ml_model(
                features.iloc[train_start-self.ml_lookback:train_start], 
                target_multiplier.iloc[train_start-self.ml_lookback:train_start]
            )
            
            # 预测自适应乘数
            adaptive_multiplier = self.predict_adaptive_multiplier(features, ml_model)
        
        print("📈 计算SuperTrend带...")
        
        # 计算SuperTrend
        for i in range(len(data)):
            if i == 0:
                continue
                
            curr_multiplier = adaptive_multiplier.iloc[i]
            curr_atr = atr.iloc[i]
            curr_hl2 = hl2.iloc[i]
            curr_close = data['close'].iloc[i]
            prev_close = data['close'].iloc[i-1]
            
            if pd.isna(curr_atr) or pd.isna(curr_hl2):
                continue
                
            # 计算上下轨
            basic_upper = curr_hl2 + curr_multiplier * curr_atr
            basic_lower = curr_hl2 - curr_multiplier * curr_atr
            
            # 上轨处理
            if i > 0 and not pd.isna(supertrend_upper.iloc[i-1]):
                if basic_upper < supertrend_upper.iloc[i-1] or prev_close > supertrend_upper.iloc[i-1]:
                    supertrend_upper.iloc[i] = basic_upper
                else:
                    supertrend_upper.iloc[i] = supertrend_upper.iloc[i-1]
            else:
                supertrend_upper.iloc[i] = basic_upper
            
            # 下轨处理
            if i > 0 and not pd.isna(supertrend_lower.iloc[i-1]):
                if basic_lower > supertrend_lower.iloc[i-1] or prev_close < supertrend_lower.iloc[i-1]:
                    supertrend_lower.iloc[i] = basic_lower
                else:
                    supertrend_lower.iloc[i] = supertrend_lower.iloc[i-1]
            else:
                supertrend_lower.iloc[i] = basic_lower
            
            # 趋势判断
            if i > 0:
                if curr_close <= supertrend_lower.iloc[i]:
                    trend.iloc[i] = -1
                elif curr_close >= supertrend_upper.iloc[i]:
                    trend.iloc[i] = 1
                else:
                    trend.iloc[i] = trend.iloc[i-1]
            
            # SuperTrend值
            if trend.iloc[i] == 1:
                supertrend.iloc[i] = supertrend_lower.iloc[i]
            else:
                supertrend.iloc[i] = supertrend_upper.iloc[i]
        
        results = {
            'supertrend': supertrend,
            'supertrend_upper': supertrend_upper,
            'supertrend_lower': supertrend_lower,
            'trend': trend,
            'adaptive_multiplier': adaptive_multiplier,
            'atr': atr
        }
        
        print("✅ ML Adaptive SuperTrend计算完成")
        return results

print("✅ MLAdaptiveSuperTrend类定义完成")

## 5. 计算指标

In [ ]:
# 检查数据
if qqq_data is not None and len(qqq_data) > 0:
    print(f"📊 开始计算QQQ的ML Adaptive SuperTrend指标...")
    print(f"数据范围: {qqq_data.index[0].date()} 到 {qqq_data.index[-1].date()}")
    
    # 创建指标实例
    ml_supertrend = MLAdaptiveSuperTrend(
        period=14,         # ATR周期
        multiplier=3.0,    # 默认乘数
        ml_lookback=50     # ML训练回看期
    )
    
    # 计算指标
    results = ml_supertrend.calculate_supertrend(qqq_data)
    
    # 将结果添加到原数据框
    for key, value in results.items():
        qqq_data[key] = value
    
    print("\n📈 指标计算完成，结果统计:")
    print(f"SuperTrend有效数据点: {qqq_data['supertrend'].count()}")
    print(f"上升趋势天数: {(qqq_data['trend'] == 1).sum()}")
    print(f"下降趋势天数: {(qqq_data['trend'] == -1).sum()}")
    print(f"平均自适应乘数: {qqq_data['adaptive_multiplier'].mean():.2f}")
    
    # 显示最新几行数据
    print("\n📋 最新数据预览:")
    display_cols = ['close', 'supertrend', 'trend', 'adaptive_multiplier']
    print(qqq_data[display_cols].tail())
    
else:
    print("❌ 数据不可用，无法计算指标")

## 6. 可视化分析

In [ ]:
def plot_ml_supertrend(data, title="ML Adaptive SuperTrend Analysis", last_days=252):
    """
    绘制ML Adaptive SuperTrend分析图
    """
    if data is None or len(data) == 0:
        print("❌ 无数据可绘制")
        return
    
    # 选择最近的数据
    plot_data = data.tail(last_days) if len(data) > last_days else data
    
    # 创建子图
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            'QQQ价格 + ML Adaptive SuperTrend',
            '自适应乘数',
            'ATR波动率'
        ),
        row_heights=[0.6, 0.2, 0.2]
    )
    
    # 主图：价格和SuperTrend
    fig.add_trace(
        go.Candlestick(
            x=plot_data.index,
            open=plot_data['open'],
            high=plot_data['high'],
            low=plot_data['low'],
            close=plot_data['close'],
            name='QQQ价格',
            increasing_line_color='green',
            decreasing_line_color='red'
        ),
        row=1, col=1
    )
    
    # SuperTrend线
    supertrend_colors = ['red' if x == -1 else 'green' for x in plot_data['trend']]
    fig.add_trace(
        go.Scatter(
            x=plot_data.index,
            y=plot_data['supertrend'],
            mode='lines',
            name='ML SuperTrend',
            line=dict(color='blue', width=2)
        ),
        row=1, col=1
    )
    
    # 添加趋势背景色
    for i in range(len(plot_data)-1):
        color = 'rgba(0,255,0,0.1)' if plot_data['trend'].iloc[i] == 1 else 'rgba(255,0,0,0.1)'
        fig.add_shape(
            type="rect",
            x0=plot_data.index[i],
            y0=plot_data['low'].min(),
            x1=plot_data.index[i+1],
            y1=plot_data['high'].max(),
            fillcolor=color,
            line=dict(width=0),
            row=1, col=1
        )
    
    # 自适应乘数图
    fig.add_trace(
        go.Scatter(
            x=plot_data.index,
            y=plot_data['adaptive_multiplier'],
            mode='lines',
            name='自适应乘数',
            line=dict(color='purple', width=2)
        ),
        row=2, col=1
    )
    
    # ATR图
    fig.add_trace(
        go.Scatter(
            x=plot_data.index,
            y=plot_data['atr'],
            mode='lines',
            name='ATR',
            line=dict(color='orange', width=2)
        ),
        row=3, col=1
    )
    
    # 更新布局
    fig.update_layout(
        title=title,
        xaxis_title="日期",
        height=800,
        showlegend=True,
        template='plotly_white'
    )
    
    fig.update_xaxes(rangeslider_visible=False)
    
    return fig

# 绘制图表
if qqq_data is not None and 'supertrend' in qqq_data.columns:
    print("📊 生成可视化图表...")
    
    # 绘制最近一年的数据
    fig = plot_ml_supertrend(qqq_data, "QQQ ML Adaptive SuperTrend分析", last_days=252)
    
    if fig is not None:
        fig.show()
        print("✅ 图表生成完成")
    else:
        print("❌ 图表生成失败")
else:
    print("❌ 数据不完整，无法生成图表")

## 7. 性能统计分析

In [ ]:
def analyze_supertrend_performance(data):
    """
    分析SuperTrend指标的性能
    """
    if data is None or 'supertrend' not in data.columns:
        print("❌ 数据不完整")
        return
    
    # 计算信号
    data['signal'] = 0
    data.loc[data['trend'] == 1, 'signal'] = 1   # 买入信号
    data.loc[data['trend'] == -1, 'signal'] = -1 # 卖出信号
    
    # 计算信号变化点
    data['signal_change'] = data['signal'].diff()
    buy_signals = data[data['signal_change'] == 2]  # 从-1变为1
    sell_signals = data[data['signal_change'] == -2] # 从1变为-1
    
    print("📊 ML Adaptive SuperTrend 性能分析")
    print("=" * 50)
    
    print(f"📈 分析期间: {data.index[0].date()} 到 {data.index[-1].date()}")
    print(f"📊 总交易日: {len(data)}")
    print(f"📊 有效信号日: {data['supertrend'].count()}")
    
    print("\n🔄 趋势统计:")
    print(f"  上升趋势天数: {(data['trend'] == 1).sum()} ({(data['trend'] == 1).sum()/len(data)*100:.1f}%)")
    print(f"  下降趋势天数: {(data['trend'] == -1).sum()} ({(data['trend'] == -1).sum()/len(data)*100:.1f}%)")
    
    print("\n📡 交易信号:")
    print(f"  买入信号次数: {len(buy_signals)}")
    print(f"  卖出信号次数: {len(sell_signals)}")
    
    if len(buy_signals) > 0:
        print(f"  最近买入信号: {buy_signals.index[-1].date()}")
    if len(sell_signals) > 0:
        print(f"  最近卖出信号: {sell_signals.index[-1].date()}")
    
    print("\n🎯 自适应参数统计:")
    print(f"  平均乘数: {data['adaptive_multiplier'].mean():.3f}")
    print(f"  乘数范围: {data['adaptive_multiplier'].min():.3f} - {data['adaptive_multiplier'].max():.3f}")
    print(f"  乘数标准差: {data['adaptive_multiplier'].std():.3f}")
    
    print("\n📈 价格与指标统计:")
    print(f"  期间收益率: {(data['close'].iloc[-1]/data['close'].iloc[0]-1)*100:.2f}%")
    print(f"  最大回撤点: {((data['close'].cummax() - data['close'])/data['close'].cummax()).max()*100:.2f}%")
    print(f"  平均ATR: {data['atr'].mean():.2f}")
    print(f"  当前价格: ${data['close'].iloc[-1]:.2f}")
    print(f"  当前SuperTrend: ${data['supertrend'].iloc[-1]:.2f}")
    print(f"  当前趋势: {'上升' if data['trend'].iloc[-1] == 1 else '下降'}")
    
    # 计算简单的持有策略收益
    if len(buy_signals) > 0 and len(sell_signals) > 0:
        print("\n💰 简单回测 (假设完美执行):")
        
        # 计算每次交易的收益
        trades = []
        for i, buy_date in enumerate(buy_signals.index):
            # 找到对应的卖出信号
            future_sells = sell_signals[sell_signals.index > buy_date]
            if len(future_sells) > 0:
                sell_date = future_sells.index[0]
                buy_price = data.loc[buy_date, 'close']
                sell_price = data.loc[sell_date, 'close']
                return_pct = (sell_price / buy_price - 1) * 100
                trades.append({
                    'buy_date': buy_date,
                    'sell_date': sell_date,
                    'buy_price': buy_price,
                    'sell_price': sell_price,
                    'return_pct': return_pct,
                    'days': (sell_date - buy_date).days
                })
        
        if trades:
            avg_return = np.mean([t['return_pct'] for t in trades])
            win_rate = len([t for t in trades if t['return_pct'] > 0]) / len(trades)
            avg_days = np.mean([t['days'] for t in trades])
            
            print(f"  总交易次数: {len(trades)}")
            print(f"  平均收益率: {avg_return:.2f}%")
            print(f"  胜率: {win_rate*100:.1f}%")
            print(f"  平均持有天数: {avg_days:.1f}天")
    
    return data

# 执行性能分析
if qqq_data is not None:
    analyzed_data = analyze_supertrend_performance(qqq_data)
else:
    print("❌ 无法进行性能分析")

## 8. 导出结果和保存数据

In [ ]:
# 保存计算结果
if qqq_data is not None and 'supertrend' in qqq_data.columns:
    try:
        # 保存完整数据
        output_file = 'QQQ_ML_SuperTrend_Results.csv'
        qqq_data.to_csv(output_file)
        print(f"✅ 完整数据已保存到: {output_file}")
        
        # 保存信号数据
        signal_data = qqq_data[qqq_data['signal_change'] != 0][['close', 'supertrend', 'trend', 'signal_change']]
        signal_file = 'QQQ_ML_SuperTrend_Signals.csv'
        signal_data.to_csv(signal_file)
        print(f"✅ 交易信号已保存到: {signal_file}")
        
        print(f"\n📁 文件保存位置: /home/user/webapp/notebooks/ml_supertrend/")
        print(f"📊 数据条数: {len(qqq_data)}")
        print(f"📅 数据期间: {qqq_data.index[0].date()} 到 {qqq_data.index[-1].date()}")
        
    except Exception as e:
        print(f"❌ 保存文件时出错: {str(e)}")

else:
    print("❌ 没有可保存的数据")

## 9. 总结和下一步

### 当前实现状态

✅ **已完成**:
1. 基础框架搭建
2. 数据获取和预处理
3. 基础SuperTrend算法实现
4. 机器学习特征工程
5. 自适应参数调整
6. 可视化分析
7. 性能统计分析

### 需要您提供Pine Script代码

⏳ **等待**:
- 请提供您的Pine Script源代码，我将:
  1. 详细分析代码逻辑
  2. 精确转换算法实现
  3. 确保计算结果的一致性
  4. 优化机器学习部分

### 准备好的功能
- ✅ QQQ数据 (2023-2025)
- ✅ 完整的可视化系统
- ✅ 性能分析框架
- ✅ 结果导出功能

**请在上方第2节中提供您的Pine Script代码，我将立即进行精确的转换和实现！**